# Normalización del dataset de Built

### 1. Importar librerias necesarias

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import os
import csv

from pathlib import Path
from difflib import get_close_matches

### 2. Carga del dataset a normalizar

In [3]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia 9.0\data\01 - Originales\07 - Built\daily_built_df.csv'

# Cargar el archivo CSV directamente
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

# Mostrar el número de municipios únicos en la columna 'municipio'
print(f"Municipios únicos en el dataset: {df['municipality_name'].nunique()}")

Filas cargadas: 2922240
Columnas disponibles: ['Unnamed: 0', 'date', 'municipality_code', 'municipality_name', 'built_fraction', 'built_area_km2', 'lat', 'geometry', 'built_growth']
Tamaño del dataset: 2922240 filas x 9 columnas


,Unnamed: 0,date,municipality_code,municipality_name,built_fraction,built_area_km2,lat,geometry,built_growth
0,13,2000-01-01,15001,Abegondo,111.620694,9169.640000,5.144827e+06,MULTIPOLYGON (((-680277.7281420507 5153036.496...,NaN
1,333,2000-01-02,15001,Abegondo,111.624217,9169.929447,5.144827e+06,MULTIPOLYGON (((-680277.7281420507 5153036.496...,NaN
2,653,2000-01-03,15001,Abegondo,111.627741,9170.218894,5.144827e+06,MULTIPOLYGON (((-680277.7281420507 5153036.496...,NaN
3,973,2000-01-04,15001,Abegondo,111.631264,9170.508342,5.144827e+06,MULTIPOLYGON (((-680277.7281420507 5153036.496...,NaN
4,1293,2000-01-05,15001,Abegondo,111.634787,9170.797789,5.144827e+06,MULTIPOLYGON (((-680277.7281420507 5153036.496...,NaN


Municipios únicos en el dataset: 317


In [ ]:
# Exportar municipios originales a un txt para normalización manual
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built'

os.makedirs(ruta_txt, exist_ok=True)
archivo_municipios = os.path.join(ruta_txt, 'municipios originales a normalizar.txt')
municipios_originales = sorted(df['municipio'].astype(str).unique())
with open(archivo_municipios, 'w', encoding='utf-8') as f:
    for m in municipios_originales:
        f.write(m + '\n')
print(f"Municipios originales exportados a: {archivo_municipios}")
print(f"Total de municipios únicos encontrados: {len(municipios_originales)}")

Municipios originales exportados a: C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\07 - Built\municipios originales a normalizar.txt


In [6]:
# Mostrar la primera fila como un diccionario columna: valor
primera_fila_dict = df.iloc[0].to_dict()
for k, v in primera_fila_dict.items():
    print(f"{k} = {v}")

Unnamed: 0 = 13
date = 2000-01-01
municipality_code = 15001
municipality_name = Abegondo
built_fraction = 111.62069385270846
built_area_km2 = 9169.64
lat = 5144827.05322505
geometry = MULTIPOLYGON (((-680277.7281420507 5153036.496481444, -680424.8126590274 5152107.984202872, -679267.4753221263 5151151.183825392, -679419.280009236 5149791.495694544, -676988.6634493698 5149242.434041278, -678215.8835869625 5148039.3901703935, -677713.8827826715 5147539.771388401, -678185.4357757979 5146763.373613848, -677689.5897442416 5146299.726065582, -678012.2423460349 5145618.947210546, -677089.2415119446 5145623.976982886, -677163.8479199182 5143815.9305977775, -677658.1177150895 5143690.42523946, -679007.4930672296 5144804.585276921, -679633.3855189213 5144556.768697833, -680262.0733112235 5144065.702664129, -679994.1643024532 5142480.246233371, -681531.1619768909 5141743.490601603, -683754.4843770997 5138929.020593785, -684679.5133144204 5139225.203039889, -685263.3585876696 5137932.364137531, -6

In [7]:
# Eliminar la columna 'geometry' (no lanza error si no existe)
df.drop(columns=['geometry'], inplace=True, errors='ignore')

In [ ]:
# Eliminar la columna 'Unnamed: 0' (índice innecesario)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print("Columna 'Unnamed: 0' eliminada correctamente")
else:
    print("ℹLa columna 'Unnamed: 0' no existe en el dataset")

print(f"Columnas restantes: {list(df.columns)}")
print(f"Nuevas dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

✅ Columna 'Unnamed: 0' eliminada correctamente
Columnas restantes: ['date', 'municipality_code', 'municipality_name', 'built_fraction', 'built_area_km2', 'lat', 'built_growth']
Nuevas dimensiones: 2922240 filas x 7 columnas


### 2.1 Normalizar los nombres de las columnas

In [ ]:
# Mostrar nombres originales de columnas
print('Nombres originales de columnas:')
print(list(df.columns))

# Normalizar nombres de columnas a español, minúsculas y descriptivos
columnas_renombrar = {
    'date': 'fecha',
    'municipality_code': 'codigo_municipio',
    'municipality_name': 'municipio',
    'built_fraction': 'fraccion_construida',
    'built_area_km2': 'area_construida_km2',
    'built_growth': 'crecimiento_construido',
    'lat': 'latitud'
}

print('\nMapeo de nombres de columnas:')
for k, v in columnas_renombrar.items():
    if k in df.columns:
        print(f'{k} -> {v}')
    else:
        print(f'{k} -> {v} (columna no encontrada)')

# Renombrar columnas que existan
columnas_existentes = {k: v for k, v in columnas_renombrar.items() if k in df.columns}
if columnas_existentes:
    df.rename(columns=columnas_existentes, inplace=True)
    print(f'\nRenombradas {len(columnas_existentes)} columnas')
else:
    print('\nNo se encontraron columnas para renombrar')

# Convertir todos los nombres a minúsculas
df.columns = [col.lower() for col in df.columns]

print('\nNombres de columnas tras la normalización:')
print(list(df.columns))

# Verificar que tenemos las columnas esperadas
columnas_esperadas = ['fecha', 'codigo_municipio', 'municipio', 'fraccion_construida', 'area_construida_km2', 'crecimiento_construido', 'latitud']
columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]
columnas_extra = [col for col in df.columns if col not in columnas_esperadas]

print(f'\nVERIFICACIÓN DE COLUMNAS:')
print(f'Columnas esperadas presentes: {len(columnas_esperadas) - len(columnas_faltantes)}/{len(columnas_esperadas)}')
if columnas_faltantes:
    print(f'Columnas faltantes: {columnas_faltantes}')
if columnas_extra:
    print(f'ℹColumnas adicionales: {columnas_extra}')

Nombres originales de columnas:
['date', 'municipality_code', 'municipality_name', 'built_fraction', 'built_area_km2', 'lat', 'built_growth']

Mapeo de nombres de columnas:
✅ date -> fecha
✅ municipality_code -> codigo_municipio
✅ municipality_name -> municipio
✅ built_fraction -> fraccion_construida
✅ built_area_km2 -> area_construida_km2
✅ built_growth -> crecimiento_construido
✅ lat -> latitud

✅ Renombradas 7 columnas

Nombres de columnas tras la normalización:
['fecha', 'codigo_municipio', 'municipio', 'fraccion_construida', 'area_construida_km2', 'latitud', 'crecimiento_construido']

📋 VERIFICACIÓN DE COLUMNAS:
✅ Columnas esperadas presentes: 7/7


### 3. Visualización y exploración inicial

In [10]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2922240 entries, 0 to 2922239
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   fecha                   object 
 1   codigo_municipio        int64  
 2   municipio               object 
 3   fraccion_construida     float64
 4   area_construida_km2     float64
 5   latitud                 float64
 6   crecimiento_construido  float64
dtypes: float64(4), int64(1), object(2)
memory usage: 156.1+ MB


### 4. Cargar el dataset limpio de municipios de Galicia

In [11]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.csv'

# Cargar el archivo CSV de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


### 5. Normalización automática de municipios

In [ ]:
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built'
os.makedirs(ruta_txt, exist_ok=True)
# Definir las columnas a usar
col_municipio = 'municipio'  # columna a normalizar en el dataset principal
col_ref = 'municipio'        # columna de referencia en el dataset de municipios
df_ref = df_municipios       # referencia oficial

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# 1. Normalizar todos los valores únicos del dataset principal
df[col_municipio] = df[col_municipio].astype(str).apply(normalizar_nombre)
municipios_unicos = set(x for x in df[col_municipio].unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

# 2. Crear mapeo: municipio original -> municipio normalizado (o sugerido, o pendiente)
mapeo = {}
pendientes = []
for m in municipios_unicos:
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        mapeo[m] = ref_norm[clave]
        continue
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[m] = ref_norm[sugerencias[0]]
        continue
    # Probar a invertir el orden de las palabras si hay exactamente dos
    partes = clave.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[m] = ref_norm[invertido]
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[m] = ref_norm[sugerencias_inv[0]]
            continue
    # Si no se encuentra nada, dejar el original y marcar como pendiente
    mapeo[m] = m
    pendientes.append(m)

# 3. Aplicar el mapeo a todo el dataset directamente sobre 'municipio'
df['municipio'] = df['municipio'].map(mapeo)

# 4. Normalización manual de municipios fusionados históricos
df['municipio'] = df['municipio'].replace({'Cesuras': 'oza-cesuras', 'Oza dos Ríos': 'oza-cesuras', 'cesuras': 'oza-cesuras', 'oza dos rios': 'oza-cesuras'})
print("Normalización manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.")

# 5. Diagnóstico de diferencias entre dataset y referencia
municipios_normalizados = set(df['municipio'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"Municipios únicos en el dataset de referencia: {len(municipios_referencia)}")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

# 6. Exportar el diccionario de correspondencias y los pendientes

archivo_diccionario = os.path.join(ruta_txt, 'diccionario_normalizacion_final.txt')
with open(archivo_diccionario, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['original', 'normalizado'])
    for k, v in sorted(mapeo.items()):
        writer.writerow([k, v])

archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in pendientes:
        f.write(f'{m}\n')

print(f"Municipios únicos originales en el dataset principal: {len(municipios_unicos)}")
print(f"Municipios normalizados automáticamente: {len(mapeo) - len(pendientes)}")
print(f"Municipios pendientes de normalizar: {len(pendientes)}")
print(f'Diccionario de normalización exportado a {archivo_diccionario}')
print(f'Listado de pendientes exportado a {archivo_pendientes}')

Normalización manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.
Municipios únicos en el dataset de referencia: 315
Municipios únicos normalizados en el dataset principal: 315
Todos los municipios de la referencia están presentes en el dataset principal.
No hay municipios extra en el dataset principal.
Municipios únicos originales en el dataset principal: 317
Municipios normalizados automáticamente: 315
Municipios pendientes de normalizar: 2
Diccionario de normalización exportado a C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built\diccionario_normalizacion_final.txt
Listado de pendientes exportado a C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built\municipios_no_normalizados_final.txt


In [ ]:
# Normalización manual de municipios fusionados históricos directamente en la columna 'municipio'

def normalizar_texto(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Normalizar todos los nombres de municipios para asegurar coincidencia
df['municipio'] = df['municipio'].apply(normalizar_texto)

print('Valores únicos antes de la sustitución manual:')
print(sorted(df['municipio'].unique()))

# Realizar la sustitución de los municipios fusionados (asegurando coincidencia exacta)
df['municipio'] = df['municipio'].replace({'cesuras': 'oza-cesuras', 'oza dos rios': 'oza-cesuras'})

print('Valores únicos después de la sustitución manual:')
print(sorted(df['municipio'].unique()))
print("Normalización manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.")

Valores únicos antes de la sustitución manual:
['a bana', 'a capela', 'a coruna', 'a laracha', 'a pobra do caraminal', 'abegondo', 'ames', 'aranga', 'ares', 'arteixo', 'arzua', 'as pontes de garcia rodriguez', 'as somozas', 'bergondo', 'betanzos', 'boimorto', 'boiro', 'boqueixon', 'brion', 'cabana de bergantinos', 'cabanas', 'camarinas', 'cambre', 'carballo', 'carnota', 'carral', 'cedeira', 'cee', 'cerceda', 'cerdido', 'coiros', 'corcubion', 'coristanco', 'culleredo', 'curtis', 'dodro', 'dumbria', 'fene', 'ferrol', 'fisterra', 'frades', 'irixoa', 'laxe', 'lousame', 'malpica de bergantinos', 'manon', 'mazaricos', 'mellid', 'mesia', 'mino', 'moeche', 'monfero', 'mugardos', 'muros', 'muxia', 'naron', 'neda', 'negreira', 'noia', 'o pino', 'oleiros', 'ordes', 'oroso', 'ortigueira', 'outes', 'oza cesuras', 'paderne', 'padron', 'ponteceso', 'pontedeume', 'porto do son', 'rianxo', 'ribeira', 'rois', 'sada', 'san sadurnino', 'santa comba', 'santiago de compostela', 'santiso', 'sobrado', 'teo', 

### 6. Exportar el dataset final con municipios normalizados

In [14]:
# Exportar el dataframe final con municipios normalizados (sin columnas duplicadas)

ruta_export = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, 'built galicia municipios normalizados.csv')

# Eliminar columnas duplicadas si las hubiera
df = df.loc[:, ~df.columns.duplicated()]

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final exportado como {archivo_export}')

Dataset final exportado como C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built\built galicia municipios normalizados.csv


In [15]:
# Comprobación final: número de municipios únicos en el CSV exportado vs referencia

csv_exportado = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built\built galicia municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)

municipios_exportados = set(df_exportado['municipio'].dropna().unique())
print(f"Municipios únicos en el CSV exportado: {len(municipios_exportados)}")
print(f"Municipios únicos en la referencia oficial: {len(municipios_referencia)}")

if municipios_exportados == municipios_referencia:
    print('¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!')
else:
    diferencia = municipios_exportados.symmetric_difference(municipios_referencia)
    print(f"Diferencias encontradas: {diferencia}")

Municipios únicos en el CSV exportado: 315
Municipios únicos en la referencia oficial: 315
¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!


In [ ]:
# Exportar los 92 municipios únicos del dataset final a un txt
ruta_export = r'C:\00 - Proyecto Incendios Galicia 9.0\data\02 - Municipio normalizado\07 - Built'
os.makedirs(ruta_export, exist_ok=True)
archivo_92_municipios = os.path.join(ruta_export, '92 municipios dataset final.txt')
municipios_finales = sorted(df['municipio'].dropna().unique())
with open(archivo_92_municipios, 'w', encoding='utf-8') as f:
    for m in municipios_finales:
        f.write(m + '\n')
print(f"Archivo exportado con los 92 municipios: {archivo_92_municipios}")

Archivo exportado con los 92 municipios: C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\07 - Built\92 municipios dataset final.txt
